# H1 versus H11a tomography

This notebook reconstructs the reported fidelities directly from the saved NetCDF acquisitions and performs an uncertainty-aware paired analysis. The bootstrap resamples each circuit's repetition index jointly across all qubits, preserving shot-level inter-qubit correlations.

The confidence intervals cover finite-shot uncertainty conditional on IQM's saved readout correction. They do not include run-to-run drift, SPAM bias, readout-calibration uncertainty, or leakage-model uncertainty.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "results").exists() else cwd.parent
RUN_DIR = ROOT / "results" / "run_20260821_181853_194798"
RAW_FILES = sorted(RUN_DIR.glob("raw_tomography_*.nc"))
if not RAW_FILES:
    raise FileNotFoundError(f"No raw tomography datasets found in {RUN_DIR}")

N_BOOTSTRAP = 5_000
RANDOM_SEED = 20260822
CI_LEVEL = 0.95

## Load the raw acquisitions

The raw NetCDF files are in the run directory. `playlists/` contains the compiled schedule visualization, not acquisition data. Each batch contains 12 tomography circuits, 100 repetitions, binary single-shot outcomes, raw classified means, and IQM-corrected excited-state probabilities.

In [ ]:
batches: dict[tuple[str, float, float], dict[str, object]] = {}
all_variable_names: set[str] = set()
inventory_rows = []

for path in RAW_FILES:
    metadata = json.loads(path.with_suffix(".json").read_text())
    qubits = tuple(metadata["qubits"])
    with xr.open_dataset(path) as opened:
        dataset = opened.load()

    all_variable_names.update(dataset.data_vars)
    single_shots = np.stack(
        [dataset[f"{q}__tomo_state_single_shot"].values for q in qubits],
        axis=-1,
    ).astype(np.int8)
    raw_probabilities = np.stack(
        [dataset[f"{q}__tomo_readout"].values for q in qubits],
        axis=-1,
    )
    corrected_probabilities = np.stack(
        [dataset[f"{q}__tomo_excited_state_probability"].values for q in qubits],
        axis=-1,
    )
    coordinates = tuple(
        (entry["input_state"], entry["measurement_basis"])
        for entry in metadata["circuit_coordinates"]
    )
    key = (
        metadata["sequence"],
        float(metadata["amplitude_error"]),
        float(metadata["detuning_hz"]),
    )
    batches[key] = {
        "path": path,
        "metadata": metadata,
        "qubits": qubits,
        "coordinates": coordinates,
        "single_shots": single_shots,
        "raw_probabilities": raw_probabilities,
        "corrected_probabilities": corrected_probabilities,
    }
    inventory_rows.append({
        "file": path.name,
        "sequence": metadata["sequence"],
        "detuning_hz": float(metadata["detuning_hz"]),
        "circuits": single_shots.shape[0],
        "shots_per_circuit": single_shots.shape[1],
        "qubits": single_shots.shape[2],
    })

inventory = pd.DataFrame(inventory_rows)
display(inventory)

reference_qubits = next(iter(batches.values()))["qubits"]
assert all(batch["qubits"] == reference_qubits for batch in batches.values())
assert all(np.isin(batch["single_shots"], (0, 1)).all() for batch in batches.values())

## Recover IQM's affine readout correction

IQM persisted both binary classified outcomes and corrected excited-state probabilities. Its correction is affine for each qubit, so the slope and intercept can be recovered from the saved pairs. During bootstrap, the correction is applied to each resampled raw mean without clipping, matching the original analysis. Calibration-parameter uncertainty cannot be recovered because those parameters were not persisted.

In [ ]:
correction_rows = []
correction_by_qubit: dict[str, tuple[float, float]] = {}
raw_mean_max_error = 0.0

for qubit_index, qubit in enumerate(reference_qubits):
    raw_values = []
    corrected_values = []
    for batch in batches.values():
        shots = batch["single_shots"][:, :, qubit_index]
        raw = batch["raw_probabilities"][:, qubit_index]
        corrected = batch["corrected_probabilities"][:, qubit_index]
        raw_mean_max_error = max(raw_mean_max_error, float(np.max(np.abs(shots.mean(axis=1) - raw))))
        raw_values.append(raw)
        corrected_values.append(corrected)

    raw_values = np.concatenate(raw_values)
    corrected_values = np.concatenate(corrected_values)
    slope, intercept = np.polyfit(raw_values, corrected_values, 1)
    residual = corrected_values - (slope * raw_values + intercept)
    correction_by_qubit[qubit] = (float(slope), float(intercept))
    correction_rows.append({
        "qubit": qubit,
        "slope": slope,
        "intercept": intercept,
        "max_abs_fit_residual": np.max(np.abs(residual)),
    })

readout_correction = pd.DataFrame(correction_rows)
print(f"Maximum difference between saved raw means and single-shot means: {raw_mean_max_error:.3e}")
print(f"Maximum affine-correction fit residual: {readout_correction['max_abs_fit_residual'].max():.3e}")
display(readout_correction)

## Reconstruct PTMs and validate the saved CSV

In [ ]:
PAULIS = (
    np.array([[1, 0], [0, 1]], dtype=complex),
    np.array([[0, 1], [1, 0]], dtype=complex),
    np.array([[0, -1j], [1j, 0]], dtype=complex),
    np.array([[1, 0], [0, -1]], dtype=complex),
)
TARGET_BLOCH = np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=float)
TARGET_PTM = np.zeros((4, 4))
TARGET_PTM[0, 0] = 1
TARGET_PTM[1:, 1:] = TARGET_BLOCH

def reconstruct_ptm(p1: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    outputs = (1 - 2 * np.asarray(p1, dtype=float)).reshape(4, 3)
    translation = (outputs[0] + outputs[1]) / 2
    bloch = np.column_stack((
        outputs[2] - translation,
        outputs[3] - translation,
        (outputs[0] - outputs[1]) / 2,
    ))
    ptm = np.zeros((4, 4))
    ptm[0, 0] = 1
    ptm[1:, 0] = translation
    ptm[1:, 1:] = bloch
    return ptm, bloch, translation

def average_gate_fidelity(ptm: np.ndarray) -> float:
    return float((np.trace(TARGET_PTM.T @ ptm) + 2) / 6)

def normalized_choi(ptm: np.ndarray) -> np.ndarray:
    choi = np.zeros((4, 4), dtype=complex)
    for column, input_pauli in enumerate(PAULIS):
        output = sum(ptm[row, column] * PAULIS[row] for row in range(4))
        choi += np.kron(input_pauli.T, output) / 4
    return (choi + choi.conj().T) / 2

point_rows = []
for (sequence, amplitude_error, detuning_hz), batch in batches.items():
    assert batch["coordinates"] == (("0", "x"), ("0", "y"), ("0", "z"), ("1", "x"), ("1", "y"), ("1", "z"), ("+x", "x"), ("+x", "y"), ("+x", "z"), ("+y", "x"), ("+y", "y"), ("+y", "z"))
    for qubit_index, qubit in enumerate(batch["qubits"]):
        ptm, bloch, translation = reconstruct_ptm(batch["corrected_probabilities"][:, qubit_index])
        fidelity = average_gate_fidelity(ptm)
        singular_values = np.linalg.svd(bloch, compute_uv=False)
        min_choi_eigenvalue = float(np.linalg.eigvalsh(normalized_choi(ptm)).min())
        point_rows.append({
            "sequence": sequence,
            "amplitude_error": amplitude_error,
            "detuning_hz": detuning_hz,
            "qubit": qubit,
            "average_gate_fidelity": fidelity,
            "max_bloch_singular_value": singular_values.max(),
            "min_choi_eigenvalue": min_choi_eigenvalue,
            "fidelity_outside_unit_interval": not (0 <= fidelity <= 1),
            "bloch_expanding": singular_values.max() > 1 + 1e-12,
        })

point_results = pd.DataFrame(point_rows)
saved_results = pd.read_csv(RUN_DIR / "composite_tomography_results.csv")
validation = point_results.merge(
    saved_results[["sequence", "amplitude_error", "detuning_hz", "qubit", "average_gate_fidelity"]],
    on=["sequence", "amplitude_error", "detuning_hz", "qubit"],
    suffixes=("_raw_reconstruction", "_saved_csv"),
)
validation["absolute_difference"] = np.abs(
    validation["average_gate_fidelity_raw_reconstruction"]
    - validation["average_gate_fidelity_saved_csv"]
)
print(f"Maximum reconstructed-versus-saved fidelity difference: {validation['absolute_difference'].max():.3e}")

In [ ]:
physicality_summary = (
    point_results.groupby(["sequence", "detuning_hz"], as_index=False)
    .agg(
        mean_fidelity=("average_gate_fidelity", "mean"),
        fidelities_outside_unit_interval=("fidelity_outside_unit_interval", "sum"),
        bloch_expanding_estimates=("bloch_expanding", "sum"),
        minimum_choi_eigenvalue=("min_choi_eigenvalue", "min"),
    )
)
display(physicality_summary)

## Joint-shot bootstrap

For each circuit and bootstrap replicate, one set of repetition indices is drawn and applied to every qubit. This retains simultaneous-shot correlations. Different circuits and acquisition batches are resampled independently because they are distinct measurements. Qubits are not resampled: the estimand is the mean performance of these fixed 20 qubits.

In [ ]:
def fidelity_from_probabilities(p1: np.ndarray) -> np.ndarray:
    expectations = 1 - 2 * p1
    m_xx = expectations[..., 6] - (expectations[..., 0] + expectations[..., 3]) / 2
    m_yz = (expectations[..., 1] - expectations[..., 4]) / 2
    m_zy = expectations[..., 11] - (expectations[..., 2] + expectations[..., 5]) / 2
    return (3 + m_xx - m_yz + m_zy) / 6

def bootstrap_batch(
    batch: dict[str, object],
    rng: np.random.Generator,
    n_bootstrap: int,
) -> np.ndarray:
    shots = batch["single_shots"]
    circuit_count, shot_count, qubit_count = shots.shape
    raw_bootstrap = np.empty((n_bootstrap, qubit_count, circuit_count))
    for circuit_index in range(circuit_count):
        indices = rng.integers(0, shot_count, size=(n_bootstrap, shot_count))
        raw_bootstrap[:, :, circuit_index] = shots[circuit_index][indices].mean(axis=1)

    slopes = np.array([correction_by_qubit[q][0] for q in batch["qubits"]])
    intercepts = np.array([correction_by_qubit[q][1] for q in batch["qubits"]])
    corrected_bootstrap = (
        raw_bootstrap * slopes[None, :, None] + intercepts[None, :, None]
    )
    return fidelity_from_probabilities(corrected_bootstrap)

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_fidelity = {
    key: bootstrap_batch(batch, rng, N_BOOTSTRAP)
    for key, batch in batches.items()
}
print(f"Generated {N_BOOTSTRAP:,} joint-shot bootstrap replicates per acquisition.")

## Paired contrasts

The absolute paired contrast is `H11a - H1` on the same qubit. The robustness advantage is

`(H1 at 0 - H1 detuned) - (H11a at 0 - H11a detuned)`.

A positive robustness advantage means H11a loses less fidelity under detuning.

In [ ]:
def interval(samples: np.ndarray, level: float = CI_LEVEL) -> tuple[float, float]:
    alpha = (1 - level) / 2
    low, high = np.quantile(samples, [alpha, 1 - alpha])
    return float(low), float(high)

amplitude_errors = sorted(point_results["amplitude_error"].unique())
if amplitude_errors != [0.0]:
    raise ValueError(f"This paired analysis expects one zero-amplitude setting, got {amplitude_errors}")
detunings = sorted(point_results["detuning_hz"].unique())
if len(detunings) != 2 or 0.0 not in detunings:
    raise ValueError(f"This paired analysis expects zero and one nonzero detuning, got {detunings}")
zero_detuning = 0.0
nonzero_detuning = next(value for value in detunings if value != 0)

point_grid = point_results.pivot(
    index="qubit",
    columns=["sequence", "detuning_hz"],
    values="average_gate_fidelity",
).loc[list(reference_qubits)]

condition_rows = []
for sequence in ("H1", "H11a"):
    for detuning_hz in detunings:
        key = (sequence, 0.0, detuning_hz)
        bootstrap_mean = bootstrap_fidelity[key].mean(axis=1)
        low, high = interval(bootstrap_mean)
        condition_rows.append({
            "sequence": sequence,
            "detuning_hz": detuning_hz,
            "mean_fidelity": point_grid[(sequence, detuning_hz)].mean(),
            "ci_low": low,
            "ci_high": high,
            "bootstrap_standard_error": bootstrap_mean.std(ddof=1),
        })
condition_summary = pd.DataFrame(condition_rows)

paired_rows = []
absolute_bootstrap: dict[float, np.ndarray] = {}
for detuning_hz in detunings:
    point_difference = point_grid[("H11a", detuning_hz)] - point_grid[("H1", detuning_hz)]
    bootstrap_difference = (
        bootstrap_fidelity[("H11a", 0.0, detuning_hz)]
        - bootstrap_fidelity[("H1", 0.0, detuning_hz)]
    )
    absolute_bootstrap[detuning_hz] = bootstrap_difference
    bootstrap_mean = bootstrap_difference.mean(axis=1)
    low, high = interval(bootstrap_mean)
    paired_rows.append({
        "contrast": "H11a - H1",
        "detuning_hz": detuning_hz,
        "mean_difference": point_difference.mean(),
        "ci_low": low,
        "ci_high": high,
    })
paired_summary = pd.DataFrame(paired_rows)

h1_drop = point_grid[("H1", zero_detuning)] - point_grid[("H1", nonzero_detuning)]
h11a_drop = point_grid[("H11a", zero_detuning)] - point_grid[("H11a", nonzero_detuning)]
robustness_by_qubit = pd.DataFrame({
    "H1_fidelity_drop": h1_drop,
    "H11a_fidelity_drop": h11a_drop,
    "H11a_robustness_advantage": h1_drop - h11a_drop,
})
robustness_bootstrap = (
    bootstrap_fidelity[("H1", 0.0, zero_detuning)]
    - bootstrap_fidelity[("H1", 0.0, nonzero_detuning)]
    - bootstrap_fidelity[("H11a", 0.0, zero_detuning)]
    + bootstrap_fidelity[("H11a", 0.0, nonzero_detuning)]
)
robustness_mean_bootstrap = robustness_bootstrap.mean(axis=1)
robustness_low, robustness_high = interval(robustness_mean_bootstrap)
robustness_summary = pd.DataFrame([{
    "contrast": "H11a robustness advantage",
    "detuning_hz": nonzero_detuning,
    "mean_difference_in_drops": robustness_by_qubit["H11a_robustness_advantage"].mean(),
    "ci_low": robustness_low,
    "ci_high": robustness_high,
    "bootstrap_standard_error": robustness_mean_bootstrap.std(ddof=1),
    "bootstrap_fraction_positive": np.mean(robustness_mean_bootstrap > 0),
}])

display(condition_summary)
display(paired_summary)
display(robustness_summary)
display(robustness_by_qubit)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for sequence in ("H1", "H11a"):
    data = condition_summary.query("sequence == @sequence").sort_values("detuning_hz")
    yerr = np.vstack((data["mean_fidelity"] - data["ci_low"], data["ci_high"] - data["mean_fidelity"]))
    axes[0].errorbar(
        data["detuning_hz"] / 1e6,
        data["mean_fidelity"],
        yerr=yerr,
        marker="o",
        capsize=4,
        linewidth=2,
        label=sequence,
    )
axes[0].set_xlabel("Pulse-local detuning (MHz)")
axes[0].set_ylabel("Linear-inversion fidelity estimate")
axes[0].set_title("Fixed-chip mean with 95% shot-bootstrap CI")
axes[0].grid(alpha=0.3)
axes[0].legend(title="Sequence")

ordered = robustness_by_qubit.sort_values("H11a_robustness_advantage")
axes[1].scatter(np.arange(len(ordered)), ordered["H11a_robustness_advantage"], color="tab:purple")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].axhline(robustness_by_qubit["H11a_robustness_advantage"].mean(), color="tab:green", linewidth=2, label="chip mean")
axes[1].axhspan(robustness_low, robustness_high, color="tab:green", alpha=0.18, label="95% shot CI")
axes[1].set_xticks(np.arange(len(ordered)), ordered.index, rotation=90)
axes[1].set_ylabel("H11a robustness advantage")
axes[1].set_title("Paired difference in fidelity drops")
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()

fig.tight_layout()
plt.show()

## Leakage identifiability

A leakage-aware analysis needs level-resolved outcomes such as calibrated `P(0)`, `P(1)`, and `P(2)`. With those data one would reconstruct a trace-non-increasing computational-subspace map, report average survival `R00`, and replace the TP formula by

`F_unconditional = (Tr(T.T @ R) + 2 * R00) / 6`.

The binary classifier saved here cannot distinguish a leaked state from whichever binary class receives it. Consequently leakage is not identifiable from this run, and a CPTP projection or clipping would not repair the missing information.

In [ ]:
leakage_keywords = ("qutrit", "leak", "p2", "state_2", "population_2", "probability_2")
leakage_resolved_variables = sorted(
    name for name in all_variable_names
    if any(keyword in name.lower() for keyword in leakage_keywords)
)
leakage_capability = pd.DataFrame([{
    "binary_single_shots_available": True,
    "level_2_resolved_variables": len(leakage_resolved_variables),
    "leakage_identifiable": bool(leakage_resolved_variables),
    "matching_variables": leakage_resolved_variables,
    "conclusion": (
        "Leakage-resolved analysis is possible."
        if leakage_resolved_variables
        else "Leakage cannot be estimated from this binary dataset."
    ),
}])
display(leakage_capability)

## Interpretation

In [ ]:
robustness_estimate = robustness_summary.iloc[0]
supported = robustness_estimate["ci_low"] > 0
display(Markdown(
    f"""
The estimated H11a robustness advantage at `{nonzero_detuning / 1e6:g} MHz` is
`{robustness_estimate['mean_difference_in_drops']:.6f}` with a
`{CI_LEVEL:.0%}` joint-shot bootstrap interval
`[{robustness_estimate['ci_low']:.6f}, {robustness_estimate['ci_high']:.6f}]`.
The finite-shot interval **{'supports' if supported else 'does not establish'}** a positive
robustness advantage. This statement remains conditional on the one run, IQM's fixed readout
calibration, the binary trace-preserving model, and uncorrected SPAM. It is not evidence about
leakage or run-to-run reproducibility.
"""
))